# Week 2 Day 5: Multi-Modal AI (Image Generation Tool) 🎨

Today we are giving our LLM the ability to "draw" images. 

**How it works:**
1. We create a tool called `generate_image`.
2. When the user asks for a picture (e.g., *"Show me a picture of a futuristic Tokyo"*), the LLM will trigger this tool.
3. Our Python tool will generate a dynamic image link using a free AI image generator (`pollinations.ai`).
4. The tool returns the link wrapped in Markdown formatting: `![Image](url)`.
5. Gradio automatically renders the Markdown as a beautiful image right inside the chat window!

In [54]:
import json
import os
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
import urllib.parse

In [55]:
# 1. Connect to model
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)
MODEL = "llama3.2:3b"

In [56]:
# 2. Define the Multi-Modal Assistant Persona
system_message = "You are a helpful multi-modal assistant. "
system_message += "You can answer questions with text, but if the user asks to see a picture, you MUST use your generate_image tool to show it to them."

In [57]:
def generate_image(prompt):
    print(f"[SYSTEM] Generating image for prompt: {prompt}")
    
    # Make the prompt safe for a web URL (e.g., changes spaces to %20)
    safe_prompt = urllib.parse.quote(prompt)
    
    # Create the free Pollinations AI image URL
    image_url = f"https://image.pollinations.ai/prompt/{safe_prompt}?width=800&height=400&nologo=true"
    
    # Return it formatted as a Markdown image so Gradio displays it as a real picture!
    return f"![Image of {prompt}]({image_url})"

In [58]:
# Describe the tool to the AI using JSON
image_tool_schema = {
    "type": "function",
    "function": {
        "name": "generate_image",
        "description": "Generates a picture or image based on a description. Call this whenever the user asks to see an image, picture, or drawing.",
        "parameters": {
            "type": "object",
            "properties": {
                "prompt": {
                    "type": "string",
                    "description": "A highly detailed visual description of the image to generate. E.g., 'A futuristic city at night with neon lights'"
                }
            },
            "required": ["prompt"]
        }
    }
}

In [59]:
tools = [image_tool_schema]


In [60]:
# Helper function to process the image tool call
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "generate_image":
            # Extract the prompt the AI wrote
            arguments = json.loads(tool_call.function.arguments)
            image_prompt = arguments.get('prompt')
            
            # Run our Python artist function
            markdown_image = generate_image(image_prompt)
            
            responses.append({
                "role": "tool",
                "content": markdown_image,
                "tool_call_id": tool_call.id
            })
    return responses

In [61]:
def chat(message, history):
    history_formatted = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history_formatted + [{"role": "user", "content": message}]
    
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        responses = handle_tool_calls(message_obj)
        messages.append(message_obj)
        messages.extend(responses)
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    # The LLM's final text summary (e.g. "Here is a picture of Tokyo...")
    final_text = response.choices[0].message.content
    
    # BUG FIX: Search our messages to see if our tool generated any images
    for msg in messages:
        # Check if the message is from our "tool"
        if isinstance(msg, dict) and msg.get("role") == "tool":
            tool_output = msg.get("content", "")
            if "![Image" in tool_output:
                # Glue the image markdown directly to the bottom of the final text!
                final_text += f"\n\n{tool_output}"
                
    return final_text

In [62]:
# Launch the Multi-Modal UI
gr.ChatInterface(fn=chat, title="Multi-Modal Artist Assistant").launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


c:\Users\ASUS\anaconda3\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


[SYSTEM] Generating image for prompt: A smiling AI assistant with a friendly and approachable expression


c:\Users\ASUS\anaconda3\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\ASUS\anaconda3\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


[SYSTEM] Generating image for prompt: A person sitting on a chair with their back straight and feet flat on the floor, with their name "Namsate" written clearly below them


c:\Users\ASUS\anaconda3\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\ASUS\anaconda3\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\ASUS\anaconda3\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\ASUS\anaconda3\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


[SYSTEM] Generating image for prompt: A majestic lion with a golden coat and a regal expression, sitting in the savannah surrounded by tall grasses and acacia trees


c:\Users\ASUS\anaconda3\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
